# Autoencoder from Scratch (PyTorch)
## Learn compression, feature extraction, and dimensionality reduction

An autoencoder learns to compress input into a lower-dimensional latent space (encoder) then reconstruct it (decoder). Used for: denoising, anomaly detection, feature learning.

---

In [ ]:
# CELL 1: Setup
import subprocess, sys
for p in ['torch','torchvision','numpy','matplotlib','seaborn','tqdm']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', p])

import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader
import torchvision, torchvision.transforms as transforms
import numpy as np, matplotlib.pyplot as plt, os
from pathlib import Path
from tqdm.auto import tqdm

# Config
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42; torch.manual_seed(SEED); np.random.seed(SEED)
OUTPUT_DIR = Path('outputs'); OUTPUT_DIR.mkdir(exist_ok=True)
BATCH_SIZE = 128; EPOCHS = 30; LATENT_DIM = 32; LR = 1e-3
print(f'Device: {device}')

In [ ]:
# CELL 2: Load MNIST Dataset
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = torchvision.datasets.MNIST(root='data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"Train: {len(train_dataset)} | Test: {len(test_dataset)} | Image shape: 28x28")

In [ ]:
# CELL 3: Autoencoder Architecture
class Autoencoder(nn.Module):
    """
    Convolutional Autoencoder for MNIST.
    Encoder: Conv2d layers compress 28x28 -> latent vector
    Decoder: ConvTranspose2d layers reconstruct latent -> 28x28
    """
    def __init__(self, latent_dim=32):
        super().__init__()
        # Encoder: 1x28x28 -> latent_dim
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1),  # -> 32x14x14
            nn.BatchNorm2d(32), nn.ReLU(True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), # -> 64x7x7
            nn.BatchNorm2d(64), nn.ReLU(True),
            nn.Flatten(),                               # -> 64*7*7 = 3136
            nn.Linear(64 * 7 * 7, latent_dim),          # -> latent_dim
            nn.ReLU(True)
        )
        # Decoder: latent_dim -> 1x28x28
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64 * 7 * 7), nn.ReLU(True),
            nn.Unflatten(1, (64, 7, 7)),
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(32), nn.ReLU(True),
            nn.ConvTranspose2d(32, 1, 3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()  # Output in [0, 1]
        )
    
    def forward(self, x):
        z = self.encoder(x)   # Encode
        x_recon = self.decoder(z)  # Decode
        return x_recon, z

model = Autoencoder(LATENT_DIM).to(device)
optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = nn.MSELoss()
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# CELL 4: Training Loop
train_losses = []
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    for batch, (images, _) in enumerate(train_loader):
        images = images.to(device)
        recon, z = model(images)
        loss = criterion(recon, images)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    if (epoch+1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {avg_loss:.6f}")

plt.plot(train_losses, 'b-', linewidth=2)
plt.title('Autoencoder Training Loss'); plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.savefig(OUTPUT_DIR / 'ae_training.png', dpi=150); plt.show()

In [ ]:
# CELL 5: Visualize Reconstructions
model.eval()
with torch.no_grad():
    test_images, _ = next(iter(test_loader))
    test_images = test_images.to(device)
    recon, latent = model(test_images)

fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for i in range(10):
    axes[0, i].imshow(test_images[i].cpu().squeeze(), cmap='gray')
    axes[1, i].imshow(recon[i].cpu().squeeze(), cmap='gray')
    axes[0, i].axis('off'); axes[1, i].axis('off')
axes[0, 0].set_ylabel('Original'); axes[1, 0].set_ylabel('Recon')
plt.suptitle('Autoencoder: Original vs Reconstructed')
plt.savefig(OUTPUT_DIR / 'ae_reconstructions.png', dpi=150); plt.show()

# Save model
torch.save(model.state_dict(), OUTPUT_DIR / 'autoencoder.pt')
print(f"Model saved! Latent dim: {LATENT_DIM}")